# Suite JRN — Journals

A journal keeps a store's history as data in a store: its journal transactions, in the order relations give them, each holding the changes one transaction made, recorded downstream of the store that made them, and read, followed, inserted into and replayed. mbse-basis [CAP-199: journals as queryable stores (mbse-journals)](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-199.md) and [CAP-288: the journal storage interface](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-288), as [the journals proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/JOURNALS.md) states.

In [ ]:
import json

from mbse.Schemas.Framework import JSON, Builders, Changes, Plain
from mbse.Journals.Framework import Journals


def crm(txn):
    """crm.Person, owned by the singleton book by key, with a name."""
    S = txn.Schemas
    return txn.uid("crm/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Holds": S.OfRelation, "Book": S.OfObject, "Person": S.OfObject})
        .spec([
            lambda _: _.Holds.name("crm.Holds").links([lambda t: t.name("book"), lambda t: t.name("contact")])
            .properties({"key": S.OfStr}).uniques([["book", "key"], ["contact"]]),
            lambda _: _.Book.name("crm.Book").ref(True).singleton("crm.TheBook").adjacencies({
                "contacts": lambda t: t.relation([_.Holds]).me("book")}),
            lambda _: _.Person.name("crm.Person").ref(True).properties({"name": S.OfStr}).adjacencies([
                lambda t: t.name("book").relation([_.Holds]).me("contact").owner(True)]),
        ])]))])


def made(install):
    """A store holding the schemas `install` defines."""
    store = Builders.Store()
    store.transaction(install)
    return store


def name_of(transaction):
    """A journal transaction's name in its journal."""
    (entry,) = transaction.journal
    return entry.name.get()


def names(journal, after=None):
    """The names of a journal's transactions, in order, from its first or after `after`."""
    return [name_of(t) for t in Journals.transactions(journal, after)]


def contacts(store):
    """The book's contacts, each its key and name, by key."""
    return sorted([e.key.get(), e.contact.name.get()] for e in store.singleton("crm.TheBook").contacts)


def add(store, *keys):
    """A transaction of `store` adding a person to its book by each key, named by it."""
    book = store.singleton("crm.TheBook")
    with store.transaction():
        for key in keys:
            store.crm.Person(lambda p, key=key: p.book(lambda a: a.insert(lambda e: e.key(key).book(book))).name(key))


model = made(crm)
journals = made(Journals.install)

<a id="JRN-01"></a>
## JRN-01 · The install defines the journals' schemas, each valid, and run again writes nothing; a journal store holds the journal's schemas alone

In [ ]:
NAMES = ["Journals.Owns", "Journals.Directory", "Journals.Contains", "Journals.First", "Journals.Last",
         "Journals.Before", "Journals.Precedes", "Journals.Binds", "Journals.Has", "Journals.Holder", "Journals.Links",
         "Journals.OfJournal", "Journals.OfTransaction", "Journals.OfBinding", "Journals.OfEntryChange",
         "Journals.OfPropertyChange"]
schemas = [journals.registered(name) for name in NAMES]
assert [schema.validate() for schema in schemas] == [[]] * len(NAMES)
journals.transaction(Journals.install)
assert [journals.registered(name) for name in NAMES] == schemas
assert "crm.Person" not in journals.names()

<a id="JRN-02"></a>
## JRN-02 · A journal, made once by its name, records each transaction a store commits as a journal transaction, appended in order, holding its changes

In [ ]:
journal = Journals.journal(journals, "crm")
assert Journals.journal(journals, "crm") is journal
stop = Journals.record(model, journal)
for key in ("ada", "bob", "cy"):
    add(model, key)
assert json.dumps(names(journal)) == '["t1", "t2", "t3"]'
first, second, third = Journals.transactions(journal)
assert json.dumps(Journals.changes(first)) == '{"mutations": [{"upsert": {"exists": [{"symbols": {"s0": {"reference": {"path": {"singleton": "crm.TheBook", "next": {"target": {}}}}}, "s1": {"schema": [{"$ref": {"singleton": ".", "next": {"entry": {"i": "schemas", "o": "", "properties": {"name": "crm"}, "next": {"entry": {"i": "schemas", "o": "", "properties": {"name": "Person"}, "next": {"target": {}}}}}}}}]}}, "spec": [{"s0": [{"as_adjacency": {"name": "contacts", "mutations": [{"insert": {"entries": [{"mutations": [{"as_link": {"name": "contact", "mutations": [{"set": {"ref": {"symbol": "s1"}}}]}}, {"as_property": {"name": "key", "mutations": [{"set": {"value": "ada"}}]}}]}]}}]}}]}, {"s1": [{"as_property": {"name": "name", "mutations": [{"set": {"value": "ada"}}]}}]}]}]}}]}'

<a id="JRN-03"></a>
## JRN-03 · Journal transactions are read in order, from the first or after any one; a journal with none reads none

In [ ]:
assert json.dumps([names(journal, first), names(journal, third), names(Journals.journal(journals, "empty"))]) == '[["t2", "t3"], [], []]'

<a id="JRN-04"></a>
## JRN-04 · A property cleared, and an entry removed by its owner's key, are held as changes and given back as written; a value is held as its plain form, the same text in both languages

In [ ]:
ada = next(e.contact for e in model.singleton("crm.TheBook").contacts if e.key.get() == "ada")
with model.transaction():
    model.crm.Person(ada, lambda p: p.name(None))
with model.transaction():
    model.crm.Book(model.singleton("crm.TheBook"), lambda b: b.contacts(lambda a: a.remove(lambda e: e.key("bob"))))
*_, cleared, removed = Journals.transactions(journal)
assert json.dumps([json.dumps(Journals.changes(cleared)), json.dumps(Journals.changes(removed))]) == '["{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"s0\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"contacts\\", \\"o\\": \\"book\\", \\"properties\\": {\\"key\\": \\"ada\\"}, \\"next\\": {\\"target\\": {}}}}}}}}, \\"spec\\": [{\\"s0\\": [{\\"as_property\\": {\\"name\\": \\"name\\", \\"mutations\\": [{\\"clear\\": {}}]}}]}]}]}}]}", "{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"s0\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"target\\": {}}}}}}, \\"spec\\": [{\\"s0\\": [{\\"as_adjacency\\": {\\"name\\": \\"contacts\\", \\"mutations\\": [{\\"remove\\": {\\"entries\\": [{\\"mutations\\": [{\\"as_property\\": {\\"name\\": \\"key\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"bob\\"}}]}}]}]}}]}}]}]}]}}]}"]'
add(model, "zoë")
*_, accented = Journals.transactions(journal)
(held,) = [e.change.value.get() for e in accented.changes if e.change.as_schema().name == "Journals.OfPropertyChange"]
assert held == '"zoë"'
with model.transaction():
    model.crm.Book(model.singleton("crm.TheBook"), lambda b: b.contacts(lambda a: a.remove(lambda e: e.key("zoë"))))

<a id="JRN-05"></a>
## JRN-05 · A journal replayed into a store holding the same schemas makes it hold what the recorded store does, each journal transaction applied in order

In [ ]:
replica = made(crm)
Journals.replay(journal, replica)
assert json.dumps([contacts(replica), contacts(model)]) == '[[["ada", null], ["cy", "cy"]], [["ada", null], ["cy", "cy"]]]'

<a id="JRN-06"></a>
## JRN-06 · A follower is given each journal transaction in order, from the first or after any one, then each appended, until it ends

In [ ]:
seen, late = [], []
end = Journals.follow(journal, lambda t: seen.append(name_of(t)))
end_late = Journals.follow(journal, lambda t: late.append(name_of(t)), third)
add(model, "dee")
end()
add(model, "eve")
end_late()
stop()
add(model, "fay")
assert json.dumps([seen, late, names(journal)]) == '[["t1", "t2", "t3", "t4", "t5", "t6", "t7", "t8"], ["t4", "t5", "t6", "t7", "t8", "t9"], ["t1", "t2", "t3", "t4", "t5", "t6", "t7", "t8", "t9"]]'

<a id="JRN-07"></a>
## JRN-07 · A journal transaction inserted after any one is read in its place, after the last appended; its changes applied again change nothing

In [ ]:
inserted = Journals.insert(journal, first, Journals.changes(first))
at_end = Journals.insert(journal, list(Journals.transactions(journal))[-1], Journals.changes(second))
again = made(crm)
Journals.replay(journal, again)
assert json.dumps([names(journal), name_of(inserted), name_of(at_end), contacts(again)]) == '[["t1", "t10", "t2", "t3", "t4", "t5", "t6", "t7", "t8", "t9", "t11"], "t10", "t11", [["ada", null], ["bob", "bob"], ["cy", "cy"], ["dee", "dee"], ["eve", "eve"]]]'

<a id="JRN-08"></a>
## JRN-08 · Two stores following one journal, each appending from what it last saw: the second append, applied where the first has, is refused, alike in both

In [ ]:
def carried(source, target, effect):
    """Applies in `target` the changes `source` reported, carried as their plain form."""
    plain = json.loads(JSON.dumps(Plain.ToPlain(source).Transaction(effect)))
    target.transaction(Plain.FromPlain(target).Transaction(plain))


here, there = made(Journals.install), made(Journals.install)
from_here, from_there = [], []
Changes.observe(here, from_here.append)
Changes.observe(there, from_there.append)
mine = Journals.journal(here, "shared")
Journals.append(mine, Journals.changes(first))
for effect in from_here:  # there holds what here does
    carried(here, there, effect)
theirs = Journals.journal(there, "shared")
Journals.append(theirs, Journals.changes(second), "there-1")  # there, after the first, as it saw it
Journals.append(mine, Journals.changes(third), "here-1")  # here, after the first too
refused = []
for source, target, effect in ((there, here, from_there[-1]), (here, there, from_here[-1])):
    try:
        carried(source, target, effect)
    except ValueError as error:
        refused.append(str(error))
assert json.dumps([refused, names(mine), names(theirs)]) == '[["an entry of \'Journals.Last\' breaks unique(transaction), where a removal from it removed nothing: entries agreeing on [\'journal\'] differ on [\'transaction\']", "an entry of \'Journals.Last\' breaks unique(transaction), where a removal from it removed nothing: entries agreeing on [\'journal\'] differ on [\'transaction\']"], ["t1", "here-1"], ["t1", "there-1"]]'